In [21]:
!pip install -q transformers datasets accelerate scikit-learn sentencepiece

Import libraries

In [22]:
import pandas as pd
import numpy as np
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.14.1+cu130
GPU available: True
GPU: Tesla T4


Load our multilingual dataset

In [23]:
multilingual_df = pd.read_csv("GWSD_Multilingual.tsv", sep="\t")

print("Dataset shape:", multilingual_df.shape)
print("\nColumns:")
print(multilingual_df.columns.tolist())

multilingual_df.head()

Dataset shape: (2300, 20)

Columns:
['Unnamed: 0', 'round', 'batch', 'sent_id', 'sentence', 'worker_0', 'worker_1', 'worker_2', 'worker_3', 'worker_4', 'worker_5', 'worker_6', 'worker_7', 'disagree', 'agree', 'neutral', 'guid', 'in_held_out_test', 'sentence_hi', 'sentence_bn']


,Unnamed: 0,round,batch,sent_id,sentence,worker_0,worker_1,worker_2,worker_3,worker_4,worker_5,worker_6,worker_7,disagree,agree,neutral,guid,in_held_out_test,sentence_hi,sentence_bn
0,0,1,0,s0,Global warming is a hoax.,disagrees,disagrees,disagrees,disagrees,disagrees,disagrees,disagrees,disagrees,NaN,NaN,NaN,1_0_s0,False,वैश्विक तापमान एक hax है.,গ্লোবাল গরম একটি হোকস।
1,1,1,0,s1,Alarming levels of sea level rise are predicte...,neutral,neutral,neutral,agrees,agrees,neutral,neutral,agrees,NaN,NaN,NaN,1_0_s1,False,समुद्र स्तर के अलार्म के स्तर अगले दशकों के दौ...,পরবর্তী দশক ধরে ফ্রিলিডিকে হুমকি দেবে নৌবাহিনী...
2,2,1,0,s2,"Over the past several years, the United States...",neutral,neutral,neutral,neutral,neutral,neutral,neutral,neutral,NaN,NaN,NaN,1_0_s2,False,"पिछले कई सालों से, अमरीका ने व्यापार वृद्धि मे...","গত কয়েক বছর ধরে, যুক্তরাষ্ট্র ব্যবসায়িক বৃদ্..."
3,3,1,0,s3,Global warming is happening and it will be dan...,agrees,agrees,agrees,agrees,agrees,agrees,agrees,agrees,NaN,NaN,NaN,1_0_s3,False,विश्‍वव्यापी तापमान हो रहा है और यह मानव स्वास...,গ্লোবাল গরমের ঘটনা ঘটেছে এবং এটি মানবাধিকার ও ...
4,4,1,0,s4,Some icebergs are cute.,neutral,neutral,neutral,neutral,neutral,neutral,neutral,neutral,NaN,NaN,NaN,1_0_s4,False,कुछ कीड़े काट रहे हैं.,কিছু আইসবেরি খুবই সুন্দর।


Prepare labels and official train/test split
We'll do exactly what we did in our baseline experiments:
- Remove 250 rows with missing aggregate stance scores.
- Convert the highest score into one of three labels.
- Preserve the official 200-row test set.
- Split the remaining 1,850 rows into 90% training and 10% validation.

In [24]:
# Remove rows with missing stance scores
score_columns = ["agree", "disagree", "neutral"]

labelled_df = multilingual_df.dropna(
    subset=score_columns
).copy()

# Create stance labels
labelled_df["stance"] = labelled_df[score_columns].idxmax(axis=1)

labelled_df["stance"] = labelled_df["stance"].map({
    "agree": "Favour",
    "disagree": "Against",
    "neutral": "Neutral"
})

# Preserve official test split
train_val_df = labelled_df[
    labelled_df["in_held_out_test"] == False
].copy()

test_df = labelled_df[
    labelled_df["in_held_out_test"] == True
].copy()

# Split training data into training and validation
train_df, val_df = train_test_split(
    train_val_df,
    test_size=0.10,
    random_state=42,
    stratify=train_val_df["stance"]
)

# Reset indexes
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Training samples:", len(train_df))
print("Validation samples:", len(val_df))
print("Testing samples:", len(test_df))

print("\nTraining distribution:")
print(train_df["stance"].value_counts())

print("\nValidation distribution:")
print(val_df["stance"].value_counts())

print("\nTest distribution:")
print(test_df["stance"].value_counts())

Training samples: 1665
Validation samples: 185
Testing samples: 200

Training distribution:
stance
Neutral    709
Favour     631
Against    325
Name: count, dtype: int64

Validation distribution:
stance
Neutral    79
Favour     70
Against    36
Name: count, dtype: int64

Test distribution:
stance
Neutral    85
Favour     76
Against    39
Name: count, dtype: int64


Convert stance labels into numerical IDs
XLM-RoBERTa needs numerical labels instead of text labels.

We'll use this mapping:
Stance	Label ID
Against	0
Favour	1
Neutral	2

In [25]:
label2id = {
    "Against": 0,
    "Favour": 1,
    "Neutral": 2
}

id2label = {
    0: "Against",
    1: "Favour",
    2: "Neutral"
}

train_df["label"] = train_df["stance"].map(label2id)
val_df["label"] = val_df["stance"].map(label2id)
test_df["label"] = test_df["stance"].map(label2id)

print("Training labels:")
print(train_df[["stance", "label"]].head())

print("\nUnique label IDs:", sorted(train_df["label"].unique()))

Training labels:
    stance  label
0   Favour      1
1  Neutral      2
2  Neutral      2
3   Favour      1
4  Neutral      2

Unique label IDs: [np.int64(0), np.int64(1), np.int64(2)]


Load the XLM-RoBERTa tokenizer

In [26]:
model_name = "FacebookAI/xlm-roberta-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Tokenizer loaded successfully!")

sample_text = train_df["sentence"].iloc[0]

encoded_sample = tokenizer(
    sample_text,
    truncation=True,
    padding="max_length",
    max_length=128
)

print("\nOriginal sentence:")
print(sample_text)

print("\nToken IDs:")
print(encoded_sample["input_ids"])

print("\nDecoded sentence:")
print(tokenizer.decode(encoded_sample["input_ids"]))

Tokenizer loaded successfully!

Original sentence:
Because climate change perceptions in children seem less susceptible to the influence of worldview or political context, it may be possible for them to inspire adults toward higher levels of climate concern, and in turn, collective action.

Token IDs:
[0, 88949, 153552, 15549, 34564, 5256, 23, 20020, 48903, 40715, 168194, 47, 70, 79507, 111, 8999, 22751, 707, 68894, 43701, 4, 442, 1543, 186, 7722, 100, 2856, 47, 16877, 13, 116127, 154732, 77546, 90926, 111, 153552, 50509, 4, 136, 23, 15504, 4, 143849, 22631, 5, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

Decoded sentence:
<s> Because climate change perceptions in children seem less susceptible to the influence of worldview or political context, it may be possible for them to inspire 

A few things from output:
- <s> = beginning of sentence.
- </s> = end of sentence.
- <pad> = padding added to make sequences equal length.
- The Hugging Face warning is normal. Authentication is optional for downloading this public model.

Tokenize English dataset

In [27]:
MAX_LENGTH = 128

train_encodings = tokenizer(
    train_df["sentence"].tolist(),
    truncation=True,
    padding="max_length",
    max_length=MAX_LENGTH
)

val_encodings = tokenizer(
    val_df["sentence"].tolist(),
    truncation=True,
    padding="max_length",
    max_length=MAX_LENGTH
)

test_encodings = tokenizer(
    test_df["sentence"].tolist(),
    truncation=True,
    padding="max_length",
    max_length=MAX_LENGTH
)

print("Training samples:", len(train_encodings["input_ids"]))
print("Validation samples:", len(val_encodings["input_ids"]))
print("Testing samples:", len(test_encodings["input_ids"]))

print("\nInput shape:", np.array(train_encodings["input_ids"]).shape)
print("Attention mask shape:", np.array(train_encodings["attention_mask"]).shape)

Training samples: 1665
Validation samples: 185
Testing samples: 200

Input shape: (1665, 128)
Attention mask shape: (1665, 128)


Convert the tokenized data into PyTorch Datasets.

In [28]:
class StanceDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.tolist()

    def __getitem__(self, idx):
        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }
        item["labels"] = torch.tensor(
            self.labels[idx],
            dtype=torch.long
        )
        return item

    def __len__(self):
        return len(self.labels)


train_dataset = StanceDataset(train_encodings, train_df["label"])
val_dataset = StanceDataset(val_encodings, val_df["label"])
test_dataset = StanceDataset(test_encodings, test_df["label"])

print("Training dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))
print("Testing dataset:", len(test_dataset))

print("\nSample keys:", train_dataset[0].keys())
print("Sample label:", train_dataset[0]["labels"])

Training dataset: 1665
Validation dataset: 185
Testing dataset: 200

Sample keys: dict_keys(['input_ids', 'attention_mask', 'labels'])
Sample label: tensor(1)


# Load the XLM-RoBERTa model

In [29]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

print("XLM-RoBERTa model loaded successfully!")
print("Number of labels:", model.config.num_labels)
print("GPU available:", torch.cuda.is_available())

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


XLM-RoBERTa model loaded successfully!
Number of labels: 3
GPU available: True


Define evaluation metrics

In [30]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)

    precision = precision_score(
        labels, predictions,
        average="macro",
        zero_division=0
    )

    recall = recall_score(
        labels, predictions,
        average="macro",
        zero_division=0
    )

    f1 = f1_score(
        labels, predictions,
        average="macro",
        zero_division=0
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

print("Evaluation metrics function created successfully!")

Evaluation metrics function created successfully!


Set training parameters

In [31]:
training_args = TrainingArguments(
    output_dir="./xlmr_english_results",

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    num_train_epochs=3,
    weight_decay=0.01,

    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    fp16=True,
    logging_steps=50,
    save_total_limit=1,

    report_to="none"
)

print("Training arguments configured successfully!")

Training arguments configured successfully!


Create the Trainer

In [32]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    processing_class=tokenizer
)

print("Trainer created successfully!")

Trainer created successfully!


Train XLM-RoBERTa on English data

In [33]:
training_output = trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.975533,0.937784,0.556757,0.697390,0.511808,0.490313
2,0.809169,0.913316,0.621622,0.668611,0.616794,0.591056
3,0.535862,0.795378,0.697297,0.705534,0.674781,0.673013


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Evaluate on the official test dataset

In [34]:
test_results = trainer.predict(test_dataset)

test_predictions = np.argmax(
    test_results.predictions,
    axis=1
)

test_true_labels = test_results.label_ids

print("English XLM-RoBERTa Test Results")
print("--------------------------------")

print("Accuracy:",
      accuracy_score(test_true_labels, test_predictions))

print("Precision:",
      precision_score(test_true_labels, test_predictions,
                      average="macro", zero_division=0))

print("Recall:",
      recall_score(test_true_labels, test_predictions,
                   average="macro", zero_division=0))

print("Macro F1:",
      f1_score(test_true_labels, test_predictions,
               average="macro", zero_division=0))

print("\nClassification Report:\n")

print(classification_report(
    test_true_labels,
    test_predictions,
    labels=[0, 1, 2],
    target_names=["Against", "Favour", "Neutral"],
    zero_division=0
))

English XLM-RoBERTa Test Results
--------------------------------
Accuracy: 0.755
Precision: 0.7442507579895663
Recall: 0.7453719139477654
Macro F1: 0.7446318281551042

Classification Report:

              precision    recall  f1-score   support

     Against       0.69      0.69      0.69        39
      Favour       0.77      0.80      0.79        76
     Neutral       0.77      0.74      0.75        85

    accuracy                           0.76       200
   macro avg       0.74      0.75      0.74       200
weighted avg       0.75      0.76      0.75       200



# Hindi XLM-RoBERTa 🇮🇳

In [35]:
MAX_LENGTH = 128

train_encodings_hi = tokenizer(
    train_df["sentence_hi"].tolist(),
    truncation=True,
    padding="max_length",
    max_length=MAX_LENGTH
)

val_encodings_hi = tokenizer(
    val_df["sentence_hi"].tolist(),
    truncation=True,
    padding="max_length",
    max_length=MAX_LENGTH
)

test_encodings_hi = tokenizer(
    test_df["sentence_hi"].tolist(),
    truncation=True,
    padding="max_length",
    max_length=MAX_LENGTH
)

print("Hindi Training samples:", len(train_encodings_hi["input_ids"]))
print("Hindi Validation samples:", len(val_encodings_hi["input_ids"]))
print("Hindi Testing samples:", len(test_encodings_hi["input_ids"]))

print("\nHindi Input shape:",
      np.array(train_encodings_hi["input_ids"]).shape)

Hindi Training samples: 1665
Hindi Validation samples: 185
Hindi Testing samples: 200

Hindi Input shape: (1665, 128)


Create Hindi PyTorch datasets

In [36]:
train_dataset_hi = StanceDataset(
    train_encodings_hi,
    train_df["label"]
)

val_dataset_hi = StanceDataset(
    val_encodings_hi,
    val_df["label"]
)

test_dataset_hi = StanceDataset(
    test_encodings_hi,
    test_df["label"]
)

print("Hindi Training dataset:", len(train_dataset_hi))
print("Hindi Validation dataset:", len(val_dataset_hi))
print("Hindi Testing dataset:", len(test_dataset_hi))

print("\nSample keys:", train_dataset_hi[0].keys())
print("Sample label:", train_dataset_hi[0]["labels"])

Hindi Training dataset: 1665
Hindi Validation dataset: 185
Hindi Testing dataset: 200

Sample keys: dict_keys(['input_ids', 'attention_mask', 'labels'])
Sample label: tensor(1)


Load a fresh XLM-RoBERTa model for Hindi

In [37]:
model_hi = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

print("Hindi XLM-RoBERTa model loaded successfully!")
print("Number of labels:", model_hi.config.num_labels)
print("GPU available:", torch.cuda.is_available())

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Hindi XLM-RoBERTa model loaded successfully!
Number of labels: 3
GPU available: True


Configure Hindi training

In [38]:
training_args_hi = TrainingArguments(
    output_dir="./xlmr_hindi_results",

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    num_train_epochs=3,
    weight_decay=0.01,

    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    fp16=True,
    logging_steps=50,
    save_total_limit=1,

    report_to="none"
)

print("Hindi training arguments configured successfully!")

Hindi training arguments configured successfully!


Hindi Trainer.

In [39]:
trainer_hi = Trainer(
    model=model_hi,
    args=training_args_hi,
    train_dataset=train_dataset_hi,
    eval_dataset=val_dataset_hi,
    compute_metrics=compute_metrics,
    processing_class=tokenizer
)

print("Hindi Trainer created successfully!")

Hindi Trainer created successfully!


# Train Hindi XLM-RoBERTa

In [40]:
training_output_hi = trainer_hi.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,1.063777,1.047714,0.400000,0.406269,0.349668,0.223955
2,1.029439,0.933866,0.529730,0.405562,0.446052,0.392091
3,0.944764,0.900268,0.621622,0.750908,0.525692,0.489642


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Evaluate Hindi on the official test set

In [41]:
test_results_hi = trainer_hi.predict(test_dataset_hi)

test_predictions_hi = np.argmax(
    test_results_hi.predictions,
    axis=1
)

test_true_labels_hi = test_results_hi.label_ids

print("Hindi XLM-RoBERTa Test Results")
print("--------------------------------")

print("Accuracy:",
      accuracy_score(test_true_labels_hi, test_predictions_hi))

print("Precision:",
      precision_score(test_true_labels_hi, test_predictions_hi,
                      average="macro", zero_division=0))

print("Recall:",
      recall_score(test_true_labels_hi, test_predictions_hi,
                   average="macro", zero_division=0))

print("Macro F1:",
      f1_score(test_true_labels_hi, test_predictions_hi,
               average="macro", zero_division=0))

print("\nClassification Report:\n")

print(classification_report(
    test_true_labels_hi,
    test_predictions_hi,
    labels=[0, 1, 2],
    target_names=["Against", "Favour", "Neutral"],
    zero_division=0
))

Hindi XLM-RoBERTa Test Results
--------------------------------
Accuracy: 0.605
Precision: 0.4060228452751817
Recall: 0.5019091847265221
Macro F1: 0.4471459855508484

Classification Report:

              precision    recall  f1-score   support

     Against       0.00      0.00      0.00        39
      Favour       0.55      0.78      0.64        76
     Neutral       0.67      0.73      0.70        85

    accuracy                           0.60       200
   macro avg       0.41      0.50      0.45       200
weighted avg       0.49      0.60      0.54       200



# Bengali XLM-RoBERTa 🇧🇩

Tokenize Bengali sentences

In [42]:
train_encodings_bn = tokenizer(
    train_df["sentence_bn"].tolist(),
    truncation=True,
    padding="max_length",
    max_length=128
)

val_encodings_bn = tokenizer(
    val_df["sentence_bn"].tolist(),
    truncation=True,
    padding="max_length",
    max_length=128
)

test_encodings_bn = tokenizer(
    test_df["sentence_bn"].tolist(),
    truncation=True,
    padding="max_length",
    max_length=128
)

print("Bengali Training samples:", len(train_encodings_bn["input_ids"]))
print("Bengali Validation samples:", len(val_encodings_bn["input_ids"]))
print("Bengali Testing samples:", len(test_encodings_bn["input_ids"]))

print("\nBengali Input shape:",
      np.array(train_encodings_bn["input_ids"]).shape)

Bengali Training samples: 1665
Bengali Validation samples: 185
Bengali Testing samples: 200

Bengali Input shape: (1665, 128)


Create Bengali PyTorch datasets

In [43]:
train_dataset_bn = StanceDataset(
    train_encodings_bn,
    train_df["label"]
)

val_dataset_bn = StanceDataset(
    val_encodings_bn,
    val_df["label"]
)

test_dataset_bn = StanceDataset(
    test_encodings_bn,
    test_df["label"]
)

print("Bengali Training dataset:", len(train_dataset_bn))
print("Bengali Validation dataset:", len(val_dataset_bn))
print("Bengali Testing dataset:", len(test_dataset_bn))

print("\nSample keys:", train_dataset_bn[0].keys())
print("Sample label:", train_dataset_bn[0]["labels"])

Bengali Training dataset: 1665
Bengali Validation dataset: 185
Bengali Testing dataset: 200

Sample keys: dict_keys(['input_ids', 'attention_mask', 'labels'])
Sample label: tensor(1)


: Load a fresh XLM-RoBERTa model for Bengali

In [44]:
model_bn = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

print("Bengali XLM-RoBERTa model loaded successfully!")
print("Number of labels:", model_bn.config.num_labels)
print("GPU available:", torch.cuda.is_available())

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Bengali XLM-RoBERTa model loaded successfully!
Number of labels: 3
GPU available: True


Configure Bengali training

In [45]:
training_args_bn = TrainingArguments(
    output_dir="./xlmr_bengali_results",

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    num_train_epochs=3,
    weight_decay=0.01,

    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    fp16=True,
    logging_steps=50,
    save_total_limit=1,

    report_to="none"
)

print("Bengali training arguments configured successfully!")

Bengali training arguments configured successfully!


Bengali Trainer

In [46]:
trainer_bn = Trainer(
    model=model_bn,
    args=training_args_bn,
    train_dataset=train_dataset_bn,
    eval_dataset=val_dataset_bn,
    compute_metrics=compute_metrics,
    processing_class=tokenizer
)

print("Bengali Trainer created successfully!")

Bengali Trainer created successfully!


# Train Bengali XLM-RoBERTa

In [47]:
training_output_bn = trainer_bn.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,1.049695,1.059496,0.405405,0.139665,0.316456,0.193798
2,1.062267,1.048028,0.427027,0.142342,0.333333,0.199495
3,1.072821,1.050998,0.427027,0.142342,0.333333,0.199495


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Evaluate Bengali on the official test set

In [48]:
test_results_bn = trainer_bn.predict(test_dataset_bn)

test_predictions_bn = np.argmax(
    test_results_bn.predictions,
    axis=1
)

test_true_labels_bn = test_results_bn.label_ids

print("Bengali XLM-RoBERTa Test Results")
print("--------------------------------")

print("Accuracy:",
      accuracy_score(test_true_labels_bn, test_predictions_bn))

print("Precision:",
      precision_score(test_true_labels_bn, test_predictions_bn,
                      average="macro", zero_division=0))

print("Recall:",
      recall_score(test_true_labels_bn, test_predictions_bn,
                   average="macro", zero_division=0))

print("Macro F1:",
      f1_score(test_true_labels_bn, test_predictions_bn,
               average="macro", zero_division=0))

print("\nClassification Report:\n")

print(classification_report(
    test_true_labels_bn,
    test_predictions_bn,
    labels=[0, 1, 2],
    target_names=["Against", "Favour", "Neutral"],
    zero_division=0
))

print("\nPrediction distribution:")
print(pd.Series(test_predictions_bn).map(id2label).value_counts())

Bengali XLM-RoBERTa Test Results
--------------------------------
Accuracy: 0.425
Precision: 0.14166666666666666
Recall: 0.3333333333333333
Macro F1: 0.19883040935672514

Classification Report:

              precision    recall  f1-score   support

     Against       0.00      0.00      0.00        39
      Favour       0.00      0.00      0.00        76
     Neutral       0.42      1.00      0.60        85

    accuracy                           0.42       200
   macro avg       0.14      0.33      0.20       200
weighted avg       0.18      0.42      0.25       200


Prediction distribution:
Neutral    200
Name: count, dtype: int64


Important research observation: XLM-RoBERTa performed very differently across languages. English performed well, Hindi struggled with Against, and Bengali collapsed to a single-class prediction. This is worth investigating and reporting honestly.

Diagnose why Bengali predictions collapsed

In [49]:
from sklearn.metrics import confusion_matrix

val_results_bn = trainer_bn.predict(val_dataset_bn)

val_predictions_bn = np.argmax(
    val_results_bn.predictions,
    axis=1
)

val_true_labels_bn = val_results_bn.label_ids

print("Bengali Validation Prediction Distribution")
print("-------------------------------------------")

print(pd.Series(val_predictions_bn).map(id2label).value_counts())

print("\nActual Validation Distribution")
print("--------------------------------")

print(pd.Series(val_true_labels_bn).map(id2label).value_counts())

print("\nConfusion Matrix")
print("----------------")

print(confusion_matrix(
    val_true_labels_bn,
    val_predictions_bn,
    labels=[0, 1, 2]
))

Bengali Validation Prediction Distribution
-------------------------------------------
Neutral    185
Name: count, dtype: int64

Actual Validation Distribution
--------------------------------
Neutral    79
Favour     70
Against    36
Name: count, dtype: int64

Confusion Matrix
----------------
[[ 0  0 36]
 [ 0  0 70]
 [ 0  0 79]]


Inspect Bengali translations

In [50]:
print("Bengali sentence samples")
print("------------------------")

for i in range(10):
    print(f"\nSentence {i+1}:")
    print("Bengali:", train_df["sentence_bn"].iloc[i])
    print("English:", train_df["sentence"].iloc[i])
    print("Stance:", train_df["stance"].iloc[i])

print("\nMissing Bengali sentences:",
      train_df["sentence_bn"].isna().sum())

print("\nEmpty Bengali sentences:",
      (train_df["sentence_bn"].astype(str).str.strip() == "").sum())

Bengali sentence samples
------------------------

Sentence 1:
Bengali: কারণ কিশোরদের ক্লামেট্রিক পরিবর্তন অনুভূতি বিশ্বব্যাপী বা রাজনৈতিক সংযোগের প্রভাবের জন্য কম প্রভাবিত মনে হয়, তাদের জন্য উচ্চতাকে উচ্চ মানের ক্লামেট্রিক উদ্বেগ এবং তারপর সামাজিক কর্মের জন্য প্রভাবিত করা সম্ভব হতে পারে।
English: Because climate change perceptions in children seem less susceptible to the influence of worldview or political context, it may be possible for them to inspire adults toward higher levels of climate concern, and in turn, collective action.
Stance: Favour

Sentence 2:
Bengali: মেটান কাটা করার সবচেয়ে কার্যকর উপায়গুলি একটি বিশ্বব্যাপী রুমিন্ট পশু, বিশেষ করে গাছের জনসংখ্যা কমাতে।
English: One of the most effective ways to cut methane is to reduce global populations of ruminant livestock, especially cattle.
Stance: Neutral

Sentence 3:
Bengali: মিনি আইসের বয়স প্রায় ১২,৮০০ বছর আগে ঘটেছিল।
English: A mini-ice age may have occurred roughly 12,800 years ago.
Stance: Neutral

Sentence 4:
Bengali: 

Check Bengali text quality

In [51]:
import re

def check_bengali_quality(text):
    text = str(text)

    bengali_chars = len(re.findall(r'[\u0980-\u09FF]', text))
    devanagari_chars = len(re.findall(r'[\u0900-\u097F]', text))
    latin_chars = len(re.findall(r'[a-zA-Z]', text))

    total = max(len(re.findall(r'\S', text)), 1)

    return pd.Series({
        "bengali_ratio": bengali_chars / total,
        "devanagari_ratio": devanagari_chars / total,
        "latin_ratio": latin_chars / total
    })

quality_df = train_df["sentence_bn"].apply(check_bengali_quality)

print("Average Bengali character ratio:",
      quality_df["bengali_ratio"].mean().round(3))

print("Average Devanagari character ratio:",
      quality_df["devanagari_ratio"].mean().round(3))

print("Average Latin character ratio:",
      quality_df["latin_ratio"].mean().round(3))

print("\nRows containing Devanagari characters:",
      (quality_df["devanagari_ratio"] > 0).sum())

Average Bengali character ratio: 0.964
Average Devanagari character ratio: 0.016
Average Latin character ratio: 0.005

Rows containing Devanagari characters: 1574


Important observation: 1,574 out of 1,665 rows contain at least one Devanagari character.

This does NOT mean 94.5% of the text is Hindi. It means those rows contain at least one Hindi-script character. The average Devanagari character ratio is only 1.6%.

So, most text is Bengali script, but mixed-script contamination is widespread.
What should we do

1. Investigate translation quality and identify the worst translations.
2. Try improving Bengali training rather than immediately abandoning XLM-RoBERTa.

Find the most contaminated Bengali sentences

In [52]:
quality_check = train_df[["sentence", "sentence_bn", "stance"]].copy()

quality_check["devanagari_ratio"] = quality_df["devanagari_ratio"].values

worst_samples = quality_check.sort_values(
    by="devanagari_ratio",
    ascending=False
).head(10)

for i, row in worst_samples.iterrows():
    print("\nEnglish:", row["sentence"])
    print("Bengali:", row["sentence_bn"])
    print("Stance:", row["stance"])
    print("Devanagari ratio:", round(row["devanagari_ratio"], 3))
    print("-" * 70)


English: We are the last generation that can prevent an unthinkable global catastrophe.
Bengali: আমরা শেষ প্রজন্ম যারা একটি অপ্রত্যাশিত বিশ্ব দুর্ঘটনাকে रोक सकते हैं।
Stance: Favour
Devanagari ratio: 0.186
----------------------------------------------------------------------

English: The expansion in Antarctic ice began to accelerate around the turn of the 21st century.
Bengali: আন্ট্রাক্টিক জেলের विस्तार ২১ শতাব্দীর দিকে দ্রুত হয়ে গিয়েছিল।
Stance: Neutral
Devanagari ratio: 0.143
----------------------------------------------------------------------

English: The Environmental Protection Agency’s broad approach to global warming is on track.
Bengali: গ্লোবাল গরমের জন্য পরিবেশ संरक्षण এজেন্সির ব্যাপক উপায় এখনই চলছে।
Stance: Neutral
Devanagari ratio: 0.14
----------------------------------------------------------------------

English: The best place to begin is air travel, with its big carbon footprint.
Bengali: শুরু করার জন্য সেরা জায়গা বিমান यात्रा, তার বড় ক্যালবোজ পতাকা।
Stanc